# Radar Financeiro V6 — processamento massivo

Preparar uma linha por MCI avaliado. O DB2 reduz a leitura transacional por partição e cliente. O Spark consolida o público e mantém as chaves das consultas seguintes no processamento distribuído.

A seção de gravação mantém o contrato de 15 campos do DDL oficial.


## 1. Conexão

> **Conexão protegida. É proibido alterar, substituir ou excluir as células de conexão.**


Use os gerenciadores carregados abaixo para acessar as fontes do Radar. O `gerenciador_spark_v2.ipynb` fornece utilitários para a sessão Spark já iniciada, e o `gerenciador_db2_spark_v2.ipynb` fornece o conector para consultas ao DB2.


In [ ]:
from traceback import format_exc

try:
    from src.utils.gerenciador_local_v2 import GerenciadorLocal

    gerenciador_local = GerenciadorLocal(
        nome_sessao="radar-todos-1",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_local.criar_sessao_spark(
        db2=True,
        driver_memory="16g",
        num_executors=6,
        executor_memory="4g",
        executor_cores=4,
        spark_conf={
            "spark.driver.memoryOverhead": "8g",
            "spark.executor.memoryOverhead": "2g",

            "spark.dynamicAllocation.enabled": "true",
            "spark.dynamicAllocation.minExecutors": "4",
            "spark.dynamicAllocation.initialExecutors": "6",
            "spark.dynamicAllocation.maxExecutors": "8",

            "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",

            "spark.speculation": "false",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
            "spark.sql.adaptive.advisoryPartitionSizeInBytes": "64m",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            "spark.sql.shuffle.partitions": "320",

            "spark.sql.broadcastTimeout": "8000",
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            "spark.sql.session.timeZone": "America/Sao_Paulo",
        },
    )

    try:
        widget_cls = __import__("ipywidgets").Widget
        ipython = get_ipython()
        if ipython is not None:
            ipython.display_formatter.formatters["text/plain"].for_type(
                widget_cls,
                lambda *a, **k: None,
            )
    except (ImportError, NameError, AttributeError):
        pass

    print("Sessão Spark inicializada.")

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


In [ ]:
%run ./src/utils/gerenciador_spark_v2.ipynb
%run ./src/utils/gerenciador_db2_spark_v2.ipynb


## 2. Parâmetros da execução

Usar `HOJE` do ambiente Spark. A quantidade de ciclos é a mesma para renda e período. O número de partições JDBC inicia em 4, como na leitura massiva de referência.

As linhas `[RADAR_METRICA]` mostram status e tempo de cada célula. `preparacao` registra a criação do plano ou da leitura JDBC. `execucao` inclui a materialização principal. As contagens aproveitam as materializações da base, do contexto e do resultado.


In [ ]:
%%spark
from contextlib import contextmanager, suppress
from datetime import date
from functools import reduce
import json
from time import perf_counter

from pyspark import StorageLevel
from pyspark.sql.window import Window

metricas_radar = {}


@contextmanager
def medir(etapa, tipo):
    inicio = perf_counter()
    dados = {"etapa": etapa, "tipo": tipo, "status": "ok"}
    print(f"[RADAR_INICIO] {etapa} ({tipo})")
    try:
        yield dados
    except (Exception, KeyboardInterrupt) as erro:
        dados["status"] = "interrompido" if isinstance(erro, KeyboardInterrupt) else "erro"
        dados["erro"] = type(erro).__name__
        raise
    finally:
        dados["segundos"] = round(perf_counter() - inicio, 3)
        metricas_radar[etapa] = dados
        print(f"[RADAR_METRICA] {json.dumps(dados, ensure_ascii=False)}")


def materializar(df, dados, anterior=None, flags=(), indicadores=()):
    df.persist(StorageLevel.MEMORY_AND_DISK)
    try:
        resultado = df.agg(
            F.count("*").alias("mcis"),
            *(F.count(F.when(F.col(flag) == "true", 1)).alias(flag) for flag in flags),
            *indicadores,
        ).first()
        dados.update(resultado.asDict())
        total_base = metricas_radar.get("Público Base", {}).get("mcis")
        if dados["etapa"] != "Público Base" and total_base is not None and dados["mcis"] != total_base:
            raise RuntimeError(f"{dados['etapa']} alterou a quantidade de MCIs: {total_base} para {dados['mcis']}.")
    except (Exception, KeyboardInterrupt):
        with suppress(Exception):
            df.unpersist()
        raise
    if anterior is not None:
        anterior.unpersist()
    return df


with medir("Parâmetros da execução", "preparacao") as dados:
    tabela_inicial = None
    dt_exea = date.fromisoformat(str(obter_variavel_ambiente('HOJE'))[:10])
    quantidade_ciclos = 1
    particoes_jdbc = 4
    conector_db2 = criar_conector_db2_spark()
    dados.update(dt_exea=dt_exea.isoformat(), ciclos=quantidade_ciclos, particoes_jdbc=particoes_jdbc)


## 3. Entrada do MCI

Primeira etapa da apresentação. A leitura de transações recentes forma o Público Base, fornece a última sincronização e identifica pessoa física. A classificação Agro complementa a base por MCI.


### Filtro inicial pela data de sincronização — Público Base

Em `DB2GFP.TRAN_RLZD_INST_PCT`, usar estado `CD_EST_TRAN_INST = 0` e a janela de um mês antes da execução até a véspera. O DB2 devolve uma linha resumida por partição física e MCI. O Spark consolida as partições e materializa uma linha por MCI, com a contagem ao final da leitura.


In [ ]:
%%spark
with medir("Público Base", "execucao") as dados:
    tabela_inicial = None
    publico_por_particao = conector_db2.sql(f"""
        SELECT
            NR_PTC,
            CD_CLI,
            MAX(TS_INCL_TRAN) AS TS_MAX,
            MAX(CASE WHEN CD_TIP_PSS = 1 THEN 1 ELSE 0 END) AS TEM_PF,
            MIN(CPF_PF) AS CPF_MIN,
            MAX(CPF_PF) AS CPF_MAX,
            MIN(AGENCIA_BB) AS AG_MIN,
            MAX(AGENCIA_BB) AS AG_MAX,
            MIN(CONTA_BB) AS CC_MIN,
            MAX(CONTA_BB) AS CC_MAX
        FROM (
            SELECT
                NR_PTC,
                CD_CLI,
                TS_INCL_TRAN,
                CD_TIP_PSS,
                CASE WHEN CD_TIP_PSS = 1 THEN NR_CPF_CNPJ_TITR END AS CPF_PF,
                CASE WHEN CD_TIP_PSS = 1 AND NR_MCA_PCT_OPB = 999999999 AND CD_PRD = 6
                          AND NR_AG_TITR IS NOT NULL AND TRIM(CD_CT_TITR) <> ''
                     THEN NR_AG_TITR
                END AS AGENCIA_BB,
                CASE WHEN CD_TIP_PSS = 1 AND NR_MCA_PCT_OPB = 999999999 AND CD_PRD = 6
                          AND NR_AG_TITR IS NOT NULL AND TRIM(CD_CT_TITR) <> ''
                     THEN TRIM(CD_CT_TITR)
                END AS CONTA_BB
            FROM DB2GFP.TRAN_RLZD_INST_PCT
            WHERE CD_CLI IS NOT NULL
              AND CD_EST_TRAN_INST = 0
              AND TS_INCL_TRAN >= TIMESTAMP(DATE('{dt_exea}') - 1 MONTH)
              AND TS_INCL_TRAN < TIMESTAMP(DATE('{dt_exea}'))
        ) MOV
        GROUP BY NR_PTC, CD_CLI
    """,
        partition_column="NR_PTC",
        lower_bound=1,
        upper_bound=101,
        num_partitions=particoes_jdbc,
    )

    publico_agregado = materializar(
        publico_por_particao
        .withColumnRenamed("CD_CLI", "CD_CLI_REF")
        .groupBy("CD_CLI_REF")
        .agg(
            F.max("TS_MAX").alias("TS_ULTIMA_INCLUSAO"),
            F.max("TEM_PF").alias("TEM_PF"),
            F.min("CPF_MIN").alias("CPF_MIN"),
            F.max("CPF_MAX").alias("CPF_MAX"),
            F.min("AG_MIN").alias("AG_MIN"),
            F.max("AG_MAX").alias("AG_MAX"),
            F.min("CC_MIN").alias("CC_MIN"),
            F.max("CC_MAX").alias("CC_MAX"),
        ),
        dados,
    )
    publico_base = publico_agregado


### Adicionar a última sincronização por MCI

Usar o maior timestamp consolidado das partições e expor o campo como `TS_INCL_TRAN_REF`.


In [ ]:
%%spark
with medir("Última sincronização por MCI", "preparacao") as dados:
    publico_base = publico_agregado.withColumnRenamed("TS_ULTIMA_INCLUSAO", "TS_INCL_TRAN_REF")


### Identificar Agro por MCI

Consultar os 11 nichos Agro aprovados no catálogo SMI. Em `DB2SMI.SUB_NHMC_CLI`, considerar o nicho principal do MCI e registrar `FLAG_AGRO` como `'true'` para os MCIs encontrados e `'false'` para os demais. A flag é usada na decisão final de participação.


In [ ]:
%%spark
with medir("Agro por MCI", "preparacao") as dados:
    agro_por_mci = conector_db2.sql("""
        SELECT DISTINCT
            CAST(agro.CD_CLI AS INTEGER) AS CD_CLI_REF,
            'true' AS FLAG_AGRO
        FROM DB2SMI.SUB_NHMC_CLI agro
        WHERE agro.CD_CLI IS NOT NULL
          AND agro.IN_NHMC_PPL_CLI = 'S'
          AND agro.NR_SEQL_NHMC IN (31, 45, 59, 60, 61, 76, 88, 125, 133, 165, 210)
    """)

    publico_base = (
        publico_base.drop("FLAG_AGRO")
        .join(agro_por_mci, "CD_CLI_REF", "left")
        .fillna("false", subset=["FLAG_AGRO"])
    )


### Identificar pessoa física

Usar o indicador já apurado na leitura inicial. `FLAG_PESSOA_FISICA` é `'true'` quando pelo menos um registro do MCI na janela tem `CD_TIP_PSS = 1`.


In [ ]:
%%spark
with medir("Pessoa física", "preparacao") as dados:
    publico_base = (
        publico_base
        .withColumn("FLAG_PESSOA_FISICA", (F.col("TEM_PF") == 1).cast("string"))
        .drop("TEM_PF")
    )


## 4. Identificação e contexto

Segunda etapa da apresentação. Cada regra enriquece o mesmo DataFrame `contexto`. Uma regra aprovada permite seguir. As flags das regras que o MCI ainda não alcançou ficam nulas.


### Identificar conta corrente BB única

Usar os mínimos e máximos já apurados para agência e conta nos registros de pessoa física. Um par preenchido é único quando os extremos coincidem nas duas colunas. Validar o formato e converter as chaves para a consulta do ciclo.


In [ ]:
%%spark
with medir("Conta corrente BB única", "preparacao") as dados:
    conta_unica = F.expr("AG_MIN = AG_MAX AND CC_MIN = CC_MAX")
    contexto = (
        publico_base
        .withColumn("CD_UOR_CC", F.when(
            conta_unica & (F.col("AG_MIN") >= 0), F.col("AG_MIN").cast("int")
        ))
        .withColumn("NR_CC", F.when(
            conta_unica & F.col("CC_MIN").rlike("^[0-9]+$"),
            F.expr("TRY_CAST(COALESCE(NULLIF(REGEXP_REPLACE(CC_MIN, '^0+', ''), ''), '0') AS DECIMAL(11,0))")
        ))
        .withColumn("FLAG_CONTA_CORRENTE_BB_UNICA", F.expr("""
            CASE WHEN FLAG_PESSOA_FISICA = 'true'
                 THEN CAST(CD_UOR_CC IS NOT NULL AND NR_CC IS NOT NULL AS STRING)
            END
        """))
        .drop("AG_MIN", "AG_MAX", "CC_MIN", "CC_MAX")
    )


### Identificar CPF único

Para os MCIs aprovados em conta BB, comparar os extremos dos CPFs preenchidos nos registros de pessoa física. Usar o CPF único para a renda e materializar o contexto antes da consulta seguinte.

A materialização inclui Agro, pessoa física, conta BB e CPF. A métrica traz os aprovados por flag e uma estimativa de CPFs distintos para dimensionar a renda.


In [ ]:
%%spark
with medir("Identificação inicial e CPF", "execucao") as dados:
    contexto = materializar(
        contexto
        .withColumn("FLAG_CPF_UNICO", F.expr("""
            CASE WHEN FLAG_CONTA_CORRENTE_BB_UNICA = 'true'
                 THEN CAST(COALESCE(CPF_MIN = CPF_MAX, FALSE) AS STRING)
            END
        """))
        .withColumn("CPF_UNICO", F.when(
            F.col("FLAG_CPF_UNICO") == "true", F.col("CPF_MIN").cast("decimal(14,0)")
        ))
        .drop("CPF_MIN", "CPF_MAX"),
        dados,
        anterior=publico_agregado,
        flags=("FLAG_AGRO", "FLAG_PESSOA_FISICA", "FLAG_CONTA_CORRENTE_BB_UNICA", "FLAG_CPF_UNICO"),
        indicadores=(F.approx_count_distinct("CPF_UNICO").alias("cpfs_distintos_estimados"),),
    )


### Consultar renda presumida

Cruzar os CPFs aprovados com a réplica Hive de `DB2DFE.REN_AVLD_PF`, como no notebook massivo. O Spark seleciona o registro mais recente por `DT_INCL_REN_AVLD`. Zero conta como renda disponível. O CPF é descartado após esse cruzamento.


In [ ]:
%%spark
with medir("Renda presumida", "execucao") as dados:
    cpfs_elegiveis = (
        contexto.where("FLAG_CPF_UNICO = 'true'")
        .select("CPF_UNICO")
        .distinct()
    )
    renda = (
        spark.table("DB2DFE.REN_AVLD_PF")
        .select(
            F.col("NR_CPF_BASE_SRF").cast("decimal(14,0)").alias("CPF_UNICO"),
            "DT_INCL_REN_AVLD", "VL_REN",
        )
        .join(cpfs_elegiveis, "CPF_UNICO", "left_semi")
        .withColumn("RN", F.row_number().over(
            Window.partitionBy("CPF_UNICO").orderBy(F.col("DT_INCL_REN_AVLD").desc_nulls_last())
        ))
        .where("RN = 1")
        .select(
            "CPF_UNICO",
            (F.col("VL_REN") * quantidade_ciclos).cast("decimal(17,2)").alias("VL_REN_PRES_REF"),
        )
    )

    contexto = materializar(
        contexto.join(renda, "CPF_UNICO", "left")
        .withColumn("FLAG_RENDA_PRESUMIDA_DISPONIVEL", F.expr("""
            CASE WHEN FLAG_CPF_UNICO = 'true'
                 THEN CAST(VL_REN_PRES_REF IS NOT NULL AS STRING)
            END
        """))
        .drop("CPF_UNICO"),
        dados,
        anterior=contexto,
        flags=("FLAG_RENDA_PRESUMIDA_DISPONIVEL",),
    )


### Consultar macroperfil

Usar os limites do público aprovado em renda para ler faixas de MCI no DB2. A origem devolve a maior `DT_REF` até a execução por MCI. O Spark mantém as chaves elegíveis e preserva a regra de interrupção quando há mais de um perfil na maior data. Os códigos 1, 2 e 3 são válidos.


In [ ]:
%%spark
def ler_db2_por_faixas(chaves, coluna_spark, coluna_db2, consulta, esquema, dados):
    minimo, maximo = chaves.agg(F.min(coluna_spark), F.max(coluna_spark)).first()
    if minimo is None:
        dados["faixas_jdbc"] = 0
        return spark.createDataFrame([], esquema)

    minimo, maximo = int(minimo), int(maximo)
    largura = (maximo - minimo + particoes_jdbc) // particoes_jdbc
    inicios = range(minimo, maximo + 1, largura)
    dados["faixas_jdbc"] = len(inicios)
    leituras = [
        conector_db2.sql(
            consulta.format(faixa=f"{coluna_db2} BETWEEN {inicio} AND {min(maximo, inicio + largura - 1)}"),
            query_timeout=3600,
        )
        for inicio in inicios
    ]
    return reduce(lambda primeira, segunda: primeira.unionByName(segunda), leituras)


with medir("Macroperfil", "execucao") as dados:
    mcis_elegiveis = contexto.where("FLAG_RENDA_PRESUMIDA_DISPONIVEL = 'true'").select("CD_CLI_REF")
    perfil = ler_db2_por_faixas(
        mcis_elegiveis, "CD_CLI_REF", "CD_CLI",
        f"""
        SELECT CAST(CD_CLI AS INTEGER) AS CD_CLI_REF,
               CAST(CD_MAC_PRFL_CLI AS INTEGER) AS CD_MAC_PRFL_CLI_REF,
               QT_PERFIS
        FROM (
            SELECT CD_CLI, CD_MAC_PRFL_CLI,
                   ROW_NUMBER() OVER (PARTITION BY CD_CLI ORDER BY DT_REF DESC) AS RN,
                   COUNT(*) OVER (PARTITION BY CD_CLI, DT_REF) AS QT_PERFIS
            FROM DB2D1D.DVS_GRDR_FNCO_PF
            WHERE DT_REF <= DATE('{dt_exea.isoformat()}') AND ({{faixa}})
        ) P
        WHERE RN = 1
        """,
        "CD_CLI_REF INT, CD_MAC_PRFL_CLI_REF INT, QT_PERFIS BIGINT", dados,
    ).join(mcis_elegiveis, "CD_CLI_REF", "left_semi")

    contexto = materializar(
        contexto.join(perfil, "CD_CLI_REF", "left")
        .withColumn("CD_MAC_PRFL_CLI_REF", F.expr("""
            CASE WHEN QT_PERFIS > 1
                 THEN CAST(raise_error('Mais de um macroperfil na maior DT_REF do MCI') AS INT)
                 ELSE CD_MAC_PRFL_CLI_REF
            END
        """))
        .drop("QT_PERFIS")
        .withColumn("FLAG_MACROPERFIL_VALIDO", F.expr("""
            CASE WHEN FLAG_RENDA_PRESUMIDA_DISPONIVEL = 'true'
                 THEN CAST(COALESCE(CD_MAC_PRFL_CLI_REF IN (1, 2, 3), FALSE) AS STRING)
            END
        """)),
        dados,
        anterior=contexto,
        flags=("FLAG_MACROPERFIL_VALIDO",),
    )


### Consultar o dia do ciclo financeiro

Ler o cadastro do ciclo por faixas de agência e cruzar os pares exatos de agência e conta dos MCIs aprovados em macroperfil. Usar o registro mais recente por `TS_ULT_EXEA_PSQ`. Manter `DD_INC_MM_CLC_BLC_REF` nulo quando não houver dia cadastrado e descartar as chaves de consulta.


In [ ]:
%%spark
with medir("Dia do ciclo", "preparacao") as dados:
    contas_elegiveis = (
        contexto.where("FLAG_MACROPERFIL_VALIDO = 'true'")
        .select("CD_UOR_CC", "NR_CC")
        .distinct()
    )
    ciclo = ler_db2_por_faixas(
        contas_elegiveis, "CD_UOR_CC", "CD_UOR_CC",
        """
        SELECT CAST(CD_UOR_CC AS INTEGER) AS CD_UOR_CC,
               CAST(NR_CC AS DECIMAL(11,0)) AS NR_CC,
               CAST(DD_INC_MM_CLC_BLC AS SMALLINT) AS DD_INC_MM_CLC_BLC_REF
        FROM (
            SELECT CD_UOR_CC, NR_CC, DD_INC_MM_CLC_BLC,
                   ROW_NUMBER() OVER (
                       PARTITION BY CD_UOR_CC, NR_CC ORDER BY TS_ULT_EXEA_PSQ DESC NULLS LAST
                   ) AS RN
            FROM DB2GFP.CT_GRDR_FNCO
            WHERE {faixa}
        ) C
        WHERE RN = 1
        """,
        "CD_UOR_CC INT, NR_CC DECIMAL(11,0), DD_INC_MM_CLC_BLC_REF SMALLINT", dados,
    ).join(contas_elegiveis, ["CD_UOR_CC", "NR_CC"], "left_semi")

    contexto_ciclo = (
        contexto.join(ciclo, ["CD_UOR_CC", "NR_CC"], "left")
        .withColumn("DD_INC_MM_CLC_BLC_REF", F.when(
            F.col("FLAG_MACROPERFIL_VALIDO") == "true", F.col("DD_INC_MM_CLC_BLC_REF")
        ))
        .drop("CD_UOR_CC", "NR_CC")
    )


## 5. Período de análise

Para os MCIs que atendem às seis condições, localizar o ciclo em andamento na data da última sincronização. O período fechado termina na véspera desse ciclo e recua pela quantidade de ciclos definida na execução. Usar o dia 1 quando faltar o dia cadastrado e ajustar o início de cada ciclo ao último dia dos meses mais curtos.


In [ ]:
%%spark
with medir("Período de análise", "preparacao") as dados:
    condicoes_participacao = F.expr("""
        FLAG_AGRO = 'true'
        AND FLAG_PESSOA_FISICA = 'true'
        AND FLAG_CONTA_CORRENTE_BB_UNICA = 'true'
        AND FLAG_CPF_UNICO = 'true'
        AND FLAG_RENDA_PRESUMIDA_DISPONIVEL = 'true'
        AND FLAG_MACROPERFIL_VALIDO = 'true'
    """)


    def inicio_ciclo_no_mes(mes, dia):
        return F.date_add(mes, F.least(dia, F.dayofmonth(F.last_day(mes))) - 1)


    data_sincronizacao = F.to_date("TS_INCL_TRAN_REF")
    mes_sincronizacao = F.trunc(data_sincronizacao, "month")
    dia_ciclo = F.coalesce(F.col("DD_INC_MM_CLC_BLC_REF").cast("int"), F.lit(1))
    inicio_no_mes = inicio_ciclo_no_mes(mes_sincronizacao, dia_ciclo)
    inicio_ciclo_aberto = F.when(data_sincronizacao >= inicio_no_mes, inicio_no_mes).otherwise(
        inicio_ciclo_no_mes(F.add_months(mes_sincronizacao, -1), dia_ciclo)
    )
    mes_inicio_periodo = F.add_months(F.trunc(inicio_ciclo_aberto, "month"), -quantidade_ciclos)

    contexto_ciclo = (
        contexto_ciclo
        .withColumn("DT_INI_REF", F.when(
            condicoes_participacao, inicio_ciclo_no_mes(mes_inicio_periodo, dia_ciclo)
        ))
        .withColumn("DT_FIM_REF", F.when(
            condicoes_participacao, F.date_sub(inicio_ciclo_aberto, 1)
        ))
    )


## 6. Montagem da tabela inicial

Depois do cálculo do período, registrar `FLAG_PARTICIPA_RADAR` como `'true'` para quem atende às seis condições e `'false'` nos demais casos. Entregar `tabela_inicial` com uma linha por MCI e os 15 campos na ordem do DDL.

A materialização inclui a leitura efetiva do ciclo e o cálculo do período. O notebook imprime `[RADAR_RESUMO]` antes da gravação. Se uma etapa falhar, use as linhas `[RADAR_METRICA]` e a mensagem do erro exibido para o diagnóstico.


In [ ]:
%%spark
with medir("Ciclo, período e tabela inicial", "execucao") as dados:
    tabela_inicial = None
    tabela_inicial = materializar(
        contexto_ciclo
        .withColumn("DT_EXEA", F.lit(dt_exea))
        .withColumn("FLAG_PARTICIPA_RADAR", F.coalesce(condicoes_participacao, F.lit(False)).cast("string"))
        .select(
            "DT_EXEA", "CD_CLI_REF", "TS_INCL_TRAN_REF", "FLAG_AGRO",
            "FLAG_PESSOA_FISICA", "FLAG_CONTA_CORRENTE_BB_UNICA", "DD_INC_MM_CLC_BLC_REF",
            "FLAG_CPF_UNICO", "FLAG_RENDA_PRESUMIDA_DISPONIVEL", "VL_REN_PRES_REF",
            "FLAG_MACROPERFIL_VALIDO", "CD_MAC_PRFL_CLI_REF",
            "DT_INI_REF", "DT_FIM_REF", "FLAG_PARTICIPA_RADAR",
        ),
        dados,
        anterior=contexto,
        flags=("FLAG_PARTICIPA_RADAR",),
        indicadores=(
            F.count("DD_INC_MM_CLC_BLC_REF").alias("mcis_com_dia_cadastrado"),
            F.count("DT_INI_REF").alias("mcis_com_periodo"),
        ),
    )

print(f"[RADAR_RESUMO] {json.dumps(list(metricas_radar.values()), ensure_ascii=False)}")


## 7. Gravação da tabela inicial

Gravar o resultado completo em `SBX_A2A13E2.ANL_RADAR_FNC_CLI`, criada pelo notebook de DDL. Esta célula substitui os dados existentes pelo resultado da execução atual.


In [ ]:
%%spark
with medir("Gravação da tabela inicial", "execucao") as dados:
    if tabela_inicial is None:
        raise RuntimeError("Execute a montagem da tabela inicial antes da gravação.")
    dados["mcis"] = metricas_radar["Ciclo, período e tabela inicial"]["mcis"]
    tabela_inicial.write.insertInto("SBX_A2A13E2.ANL_RADAR_FNC_CLI", overwrite=True)
